# Demo 2 · Un tablero de control con KPIs: del registro de eventos a las decisiones
**Analítica de procesos para ciberseguridad** · demostración de unos 15 minutos

**La idea.** En la Práctica 2 reconstruimos *cómo* se atiende un incidente. Un equipo de seguridad (o de TI) no vive mirando mapas de proceso: vive mirando un **tablero** con unos pocos números que le dicen **si el proceso va bien**. Esos números se llaman **KPI** (*Key Performance Indicators*, indicadores clave de desempeño). Hoy construimos uno con los **mismos datos de la Práctica 2**.

**Qué es un buen tablero:**
- **Pocos indicadores** (aquí, 8), cada uno con una definición clara y una **meta**.
- **Equilibrado**: no solo mide **velocidad**, también mide **control y calidad** del proceso. Veremos por qué.
- Con **contexto**: cómo cambia en el tiempo y cómo se reparte (por prioridad, por grupo).

**Los 8 KPI de esta demo**

| Familia | KPI | Qué cuenta |
|---|---|---|
| Resultado | **Incidentes** | Cuántos incidentes hay en la vista |
| Resultado | **Cumplen el SLA** | % de incidentes que cumplieron el plazo acordado |
| Resultado | **Mediana para resolver** | Días desde que se abre el incidente hasta que se resuelve (la mitad tarda menos) |
| Resultado | **Percentil 90 para resolver** | Días en que se resuelve el 90 % (muestra los casos lentos) |
| Proceso | **Reabiertos** | % de incidentes que se reabrieron después de resueltos |
| Proceso | **Rebotan entre grupos** | % de incidentes que vuelven a un grupo por el que ya habían pasado (A → B → A) |
| Proceso | **Se saltan «En curso»** | % de incidentes que se resuelven sin pasar por el estado de trabajo |
| Proceso | **Esperan al usuario** | % de incidentes que se detuvieron esperando información del usuario |

**Los datos.** El mismo registro de la Práctica 2: una mesa de servicio de **TI** (ServiceNow), 24.916 incidentes y 141.697 eventos. No es un SOC de seguridad, pero un incidente de seguridad se gestiona igual.

> **Fuente y cambios.** Amaral, Fantinato, Reijers y Peres (2018). *Incident management process enriched event log.* UCI Machine Learning Repository, https://doi.org/10.24432/C57S4H. Licencia CC BY 4.0. Cambios hechos con fines académicos (ver `datos/LEEME_datos.md`).

## 0 · Preparación

In [ ]:
# Importa las librerías: pandas (tablas), numpy (cálculos) y matplotlib (gráficos)
import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import display
warnings.filterwarnings("ignore")                # oculta advertencias para que la salida se lea limpia
pd.set_option("display.max_colwidth", 70); pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

# Dirección de los datos: repositorio de GitHub y versión fija de donde se descargan los archivos
REPO = "lsandinop/analitica-procesos-ciberseguridad"
VERSION = "main"
DATA_BASE = "" if REPO.startswith("USUARIO") else f"https://raw.githubusercontent.com/{REPO}/{VERSION}/datos/"

def cargar(nombre, fechas=None):
    # Lee un archivo CSV y lo devuelve como tabla (DataFrame). Lo busca primero en GitHub;
    # si no está disponible, en la carpeta local. `fechas` son las columnas que se leen como fecha y hora.
    if DATA_BASE:
        try:
            return pd.read_csv(DATA_BASE + nombre, parse_dates=fechas)
        except Exception:
            print("No pude descargar el archivo de GitHub; busco una copia local.")
    for ruta in (nombre, "datos/" + nombre, "../datos/" + nombre, "../../datos/" + nombre):
        if os.path.exists(ruta):
            return pd.read_csv(ruta, parse_dates=fechas)
    raise FileNotFoundError(f"No encuentro {nombre}: súbelo a la carpeta del cuaderno")

eventos = cargar("incidentes_servicenow.csv", fechas=["fecha_evento", "fecha_apertura", "fecha_resolucion", "fecha_cierre"])
print(f"{len(eventos):,} filas (eventos) · {eventos.incidente.nunique():,} incidentes".replace(",", "."))

## 1 · De eventos a una tabla con un incidente por fila
El registro tiene **una fila por evento**. Un tablero habla de **incidentes**, así que primero resumimos cada incidente en **una fila**, con lo que necesitan los KPI (la misma lógica de la Práctica 2):

In [ ]:
eventos = eventos.sort_values(["incidente", "fecha_evento"], kind="stable")

# La ruta de cada incidente: sus estados, en orden, sin repetir el mismo estado dos veces seguidas
cambio = eventos.estado != eventos.groupby("incidente").estado.shift()
pasos = eventos.loc[cambio].groupby("incidente").estado.apply(list)

# Los grupos de trabajo por los que pasó cada incidente, en orden y sin repetir seguidos
grupos = eventos.groupby("incidente").grupo_asignado.apply(lambda s: s[s != s.shift()].tolist())

def rebota(secuencia):
    # ¿La secuencia vuelve a donde estaba dos pasos antes (A → B → A)? Es un «rebote» o ping-pong
    return any(secuencia[i] == secuencia[i - 2] for i in range(2, len(secuencia)))

# Una fila por incidente
casos = eventos.groupby("incidente").agg(apertura=("fecha_apertura", "first"), resolucion=("fecha_resolucion", "last"), prioridad=("prioridad", "first"),
                                         categoria=("categoria", "first"), grupo=("grupo_asignado", "first"), reaperturas=("reaperturas", "max"),
                                         cumplio_sla=("cumplio_sla", "last"))
casos["dias_a_resolver"] = (casos.resolucion - casos.apertura).dt.total_seconds() / 86400     # días desde la apertura hasta la resolución
casos["reabierto"] = casos.reaperturas > 0
casos["rebota_entre_grupos"] = grupos.reindex(casos.index).map(rebota)
casos["salta_en_curso"] = pasos.reindex(casos.index).map(lambda p: "En curso" not in p)
casos["espera_al_usuario"] = pasos.reindex(casos.index).map(lambda p: "Esperando al usuario" in p)

print(f"{len(casos):,} incidentes · {casos.dias_a_resolver.isna().sum():,} no tienen fecha de resolución: los KPI de tiempo usan los {casos.dias_a_resolver.notna().sum():,} que sí".replace(",", "."))
casos.head(5)

## 2 · Los KPI y su semáforo
Una función calcula los 8 KPI para **cualquier grupo de incidentes** (todos, solo los críticos, solo un equipo...). Así el mismo tablero sirve para todas las vistas. Cada KPI tiene una **meta** con tres colores:
- **verde**: dentro de la meta · **ámbar**: en alerta · **rojo**: fuera de la meta.

> **Las metas de esta demo son ejemplos** puestos para ilustrar el semáforo. En una organización real las fija el negocio, no los datos.

In [ ]:
def calcular_kpis(c):
    # Calcula los 8 KPI de un grupo de incidentes `c` (una fila por incidente)
    resueltos = c.dias_a_resolver.dropna()                       # los tiempos solo se calculan con incidentes que tienen fecha de resolución
    return {"Incidentes": len(c),
            "Cumplen el SLA": c.cumplio_sla.mean() * 100,
            "Mediana para resolver": resueltos.median(),
            "Percentil 90 para resolver": resueltos.quantile(0.9),
            "Reabiertos": c.reabierto.mean() * 100,
            "Rebotan entre grupos": c.rebota_entre_grupos.mean() * 100,
            "Se saltan «En curso»": c.salta_en_curso.mean() * 100,
            "Esperan al usuario": c.espera_al_usuario.mean() * 100}

# Metas de EJEMPLO: (sentido, verde hasta/desde, ámbar hasta/desde). «más» = más es mejor; «menos» = menos es mejor
METAS = {"Cumplen el SLA": ("más", 90, 75), "Mediana para resolver": ("menos", 1, 2), "Percentil 90 para resolver": ("menos", 5, 10),
         "Reabiertos": ("menos", 2, 5), "Rebotan entre grupos": ("menos", 3, 6), "Se saltan «En curso»": ("menos", 10, 25), "Esperan al usuario": ("menos", 15, 30)}
UNIDAD = {"Incidentes": "", "Cumplen el SLA": "%", "Mediana para resolver": "días", "Percentil 90 para resolver": "días",
          "Reabiertos": "%", "Rebotan entre grupos": "%", "Se saltan «En curso»": "%", "Esperan al usuario": "%"}

def color_semaforo(nombre, valor):
    # Verde, ámbar o rojo según la meta del KPI (gris si el KPI no tiene meta o no hay datos)
    if nombre not in METAS or pd.isna(valor):
        return "#a8b0b6"
    sentido, verde, ambar = METAS[nombre]
    if sentido == "más":
        return "#5cb85c" if valor >= verde else "#f0ad4e" if valor >= ambar else "#d9534f"
    return "#5cb85c" if valor <= verde else "#f0ad4e" if valor <= ambar else "#d9534f"

def texto_valor(nombre, valor):
    # El número tal como se lee en español: «63,4 %», «0,9 días», «24.916»
    if pd.isna(valor):
        return "—"
    if nombre == "Incidentes":
        return f"{valor:,.0f}".replace(",", ".")
    return f"{valor:.1f}".replace(".", ",") + (" %" if UNIDAD[nombre] == "%" else " días")

def tarjetas(kpis, titulo):
    # Dibuja el tablero: una tarjeta por KPI, con el número grande y una franja del color del semáforo
    fig, ejes = plt.subplots(2, 4, figsize=(14, 4.6))
    for ax, (nombre, valor) in zip(ejes.ravel(), kpis.items()):
        ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
        ax.add_patch(Rectangle((0, 0), 1, 1, fc="#f6f4ee", ec="#dfdfdb"))
        ax.add_patch(Rectangle((0, 0.9), 1, 0.1, fc=color_semaforo(nombre, valor), ec="none"))
        ax.text(0.5, 0.5, texto_valor(nombre, valor), ha="center", va="center", fontsize=22, fontweight="bold", color="#0f2437")
        ax.text(0.5, 0.15, nombre, ha="center", va="center", fontsize=10, color="#5a676f")
        if nombre in METAS:
            sentido, verde, ambar = METAS[nombre]
            unidad = "día" if (UNIDAD[nombre] == "días" and verde == 1) else UNIDAD[nombre]
            ax.text(0.5, 0.78, f"meta: {'al menos' if sentido == 'más' else 'hasta'} {verde} {unidad}", ha="center", va="center", fontsize=8, color="#5a676f")
    fig.suptitle(titulo, fontsize=12, fontweight="bold", color="#0f2437", x=0.01, ha="left")
    plt.tight_layout(); plt.show()

kpis_total = calcular_kpis(casos)
tarjetas(kpis_total, "Tablero de la mesa de servicio · todos los incidentes")

**Cómo leerlo.** Cada tarjeta es un KPI. La franja de arriba es el semáforo (verde, ámbar, rojo) frente a la meta de ejemplo. Aquí **cuatro KPI están en rojo**: el cumplimiento del plazo, los casos lentos, los rebotes entre grupos y los saltos de *En curso*. La **mediana** para resolver, en cambio, está en verde: la mitad de los incidentes se resuelve en menos de un día. **Un solo KPI nunca cuenta toda la historia**: por eso se miran varios a la vez y en el tiempo.

## 3 · ¿Cómo cambia en el tiempo?
Un número suelto no dice si el proceso mejora o empeora. Miramos tres KPI **por semana de apertura** del incidente. (Casi todos los incidentes del registro se abrieron entre marzo y mayo de 2016; mostramos solo las semanas con **100 incidentes o más**, porque con menos la cifra se mueve demasiado).

In [ ]:
casos["semana"] = casos.apertura.dt.to_period("W-SUN").dt.start_time              # el lunes de la semana en que se abrió el incidente
semanal = casos.groupby("semana").agg(incidentes=("cumplio_sla", "size"), sla=("cumplio_sla", "mean"), mediana=("dias_a_resolver", "median"), salta=("salta_en_curso", "mean"))
semanal = semanal[semanal.incidentes >= 100]

fig, ejes = plt.subplots(1, 3, figsize=(15, 3.8))
series = [("sla", "% de incidentes que cumplen el SLA", lambda s: s * 100, "#1f6db0"), ("mediana", "Mediana de días para resolver", lambda s: s, "#b24e14"),
          ("salta", "% que se salta «En curso»", lambda s: s * 100, "#5a676f")]
for ax, (col, titulo, f, color) in zip(ejes, series):
    ax.plot(semanal.index, f(semanal[col]), "o-", color=color)
    ax.set_title(titulo, fontsize=10); ax.tick_params(axis="x", rotation=40, labelsize=8); ax.set_ylim(bottom=0)
    ax.grid(axis="y", alpha=0.3)
fig.suptitle(f"Tres KPI por semana de apertura ({len(semanal)} semanas con 100 incidentes o más)", fontsize=11, x=0.01, ha="left")
plt.tight_layout(); plt.show()

**Qué leer ahí.** Algo cambió en marzo de 2016: el **cumplimiento del plazo cae a 4 %** en la semana del 7 de marzo y **sube a entre 66 % y 82 %** desde abril, y la **mediana para resolver baja de varios días a menos de medio día**. Mientras tanto, el porcentaje de incidentes que se saltan *En curso* **casi no se mueve**. El tablero **muestra que algo cambió, no por qué**: ¿se modificó el proceso?, ¿se limpió un rezago acumulado?, ¿cambió la forma de registrar? Responderlo exige preguntarle a quien conoce el negocio.

## 4 · ¿Cómo se reparte? Por prioridad y por equipo
Un promedio general puede esconder grupos que van muy distinto. Desglosamos el cumplimiento del plazo por **prioridad** y por **grupo de trabajo**:

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 3.8), gridspec_kw={"width_ratios": [1, 1.25]})

# (a) Por prioridad
por_prioridad = casos.groupby("prioridad").agg(incidentes=("cumplio_sla", "size"), sla=("cumplio_sla", "mean"), mediana=("dias_a_resolver", "median"))
por_prioridad["sla"] *= 100
ax = ejes[0]; ax.barh(por_prioridad.index[::-1], por_prioridad.sla[::-1], color=[color_semaforo("Cumplen el SLA", v) for v in por_prioridad.sla[::-1]])
for i, (v, n) in enumerate(zip(por_prioridad.sla[::-1], por_prioridad.incidentes[::-1])):
    ax.text(v + 1, i, f"{v:.0f} %  ({n:,} incidentes)".replace(",", "."), va="center", fontsize=9)
ax.set_xlim(0, 135); ax.set_xlabel("% de incidentes que cumplen el SLA"); ax.set_title("(a) Cumplimiento del plazo por prioridad", fontsize=10)

# (b) Por grupo de trabajo: los 8 con peor cumplimiento entre los que tienen 300 incidentes o más
por_grupo = casos.groupby("grupo").agg(incidentes=("cumplio_sla", "size"), sla=("cumplio_sla", "mean"), mediana=("dias_a_resolver", "median"))
por_grupo = por_grupo[por_grupo.incidentes >= 300].sort_values("sla").head(8)
por_grupo["sla"] *= 100
ax = ejes[1]; ax.barh(por_grupo.index[::-1], por_grupo.sla[::-1], color=[color_semaforo("Cumplen el SLA", v) for v in por_grupo.sla[::-1]])
for i, (v, n, m) in enumerate(zip(por_grupo.sla[::-1], por_grupo.incidentes[::-1], por_grupo.mediana[::-1])):
    n_txt, m_txt = f"{n:,}".replace(",", "."), f"{m:.1f}".replace(".", ",")                  # miles con punto y decimales con coma
    ax.text(v + 1, i, f"{v:.0f} %  ({n_txt} incidentes · mediana {m_txt} días)", va="center", fontsize=8)
ax.set_xlim(0, 135); ax.set_xlabel("% de incidentes que cumplen el SLA"); ax.set_title("(b) Los 8 grupos con peor cumplimiento (300 incidentes o más)", fontsize=10)
plt.tight_layout(); plt.show()

**Qué leer ahí.**
- **(a)** Lo **más urgente es lo que más incumple**: los incidentes de prioridad *crítica* cumplen el plazo solo el 19 % de las veces y los de prioridad *alta*, el 3 %; los *moderados*, el 64 %, y los *bajos*, el 85 %. El promedio general (63 %) lo oculta porque casi todo el volumen es *moderado*. **Un KPI promedio puede esconder justo lo que más importa.** Ojo: no sabemos *por qué* (quizá el plazo de esas prioridades es más estricto): hay que preguntarlo, no suponerlo.
- **(b)** Algunos grupos concentran el problema: los de abajo cumplen el plazo menos de la mitad de las veces, y alguno tarda semanas (mediana) en resolver. Un tablero sirve para decir **dónde conversar**.

## 5 · El tablero, filtrado
Un tablero útil se puede **filtrar**. Esta función dibuja las mismas 8 tarjetas para el grupo de incidentes que elijas. Primero, un ejemplo fijo: solo los incidentes **críticos**.

In [ ]:
def filtrar(prioridad="Todas", categoria="Todas"):
    # Dibuja el tablero solo con los incidentes de la prioridad y la categoría elegidas
    c = casos
    if prioridad != "Todas":
        c = c[c.prioridad == prioridad]
    if categoria != "Todas":
        c = c[c.categoria == categoria]
    if len(c) == 0:
        print("No hay incidentes con esa combinación.")
        return
    tarjetas(calcular_kpis(c), f"Prioridad: {prioridad}  ·  Categoría: {categoria}  ·  {len(c):,} incidentes".replace(",", "."))

filtrar("1 - Crítica")

**Ahora, el tuyo.** Cambia los menús y mira cómo se mueven los KPI (en Colab; si no aparecen los menús, ejecuta otra vez la celda). Busca: ¿qué combinación tiene el peor cumplimiento del plazo? ¿Y la peor mediana?

In [ ]:
try:
    import ipywidgets as widgets
    prioridades = ["Todas"] + sorted(casos.prioridad.unique())
    categorias = ["Todas"] + list(casos.categoria.value_counts().head(8).index)          # las 8 categorías con más incidentes
    widgets.interact(filtrar, prioridad=prioridades, categoria=categorias)
except ImportError:
    print("Esta celda necesita la librería ipywidgets (en Google Colab ya viene instalada).")

## 6 · El indicador que engaña: cuando la velocidad premia el atajo
En la Práctica 2 vimos que casi 4 de cada 10 incidentes se resuelven **sin pasar por «En curso»**. ¿Cómo se ven los KPI de velocidad en esos incidentes, frente a los que sí siguen el paso?

In [ ]:
comparacion = pd.DataFrame({"Se saltan «En curso»": calcular_kpis(casos[casos.salta_en_curso]), "Pasan por «En curso»": calcular_kpis(casos[~casos.salta_en_curso])})
display(comparacion.loc[["Incidentes", "Cumplen el SLA", "Mediana para resolver", "Percentil 90 para resolver"]].round(1))

**Qué leer ahí.** Los incidentes que **se saltan «En curso» cumplen mejor el plazo y se resuelven más rápido**. Para un tablero que solo mira velocidad, saltarse el paso **parece una buena práctica**: el atajo mejora los KPI. Pero en seguridad ese paso puede ser justo la **investigación**. Por eso un tablero equilibrado mira a la vez **resultado** (velocidad, plazos) y **proceso** (saltos, rebotes, reaperturas): si solo mides lo primero, premias el atajo. Y recuerda la regla de la minería de procesos: los KPI muestran **dónde mirar**, no **por qué** pasa.

## Para pensar
1. **Si fueras el jefe del equipo, ¿qué KPI agregarías o quitarías para que el tablero no se pueda «ganar» con atajos?** ¿Cómo evitarías que la gente los trabaje solo para ponerlos en verde?
2. **Los incidentes de prioridad alta cumplen el plazo solo el 3 % de las veces.** ¿Qué hipótesis tienes y qué datos pedirías para probarla antes de actuar?
3. **Las metas de este tablero son de ejemplo.** ¿Quién debería fijar la meta de cumplimiento del plazo, cada cuánto se revisa y quién debería ver este tablero?

## Cierre
- Un **KPI** es un número con **definición y meta**; un **tablero** junta pocos, **equilibrados** (resultado y proceso) y **con contexto** (tiempo y desgloses).
- Los KPI salen de **tablas de incidentes** que se construyen a partir del **registro de eventos**: lo mismo que hicimos en la Práctica 2.
- Un promedio puede **esconder** lo importante (la prioridad alta); un indicador de velocidad puede **premiar el atajo**.
- El tablero señala **dónde investigar**, no la causa.